# 13.6 实时语音双工 (Realtime Speech Duplex)

> 🕐 预估学习时间：40分钟

实时语音助手需要边听边说、可打断（barge-in）、低尾音延迟。级联 ASR→LLM→TTS 与原生全双工端到端是两条主流路线。

本节涵盖：
- 延迟预算拆解
- 端点检测 / 打断 / 回声
- 流式 ASR 部分假设 → 投机回复
- 半双工状态机 vs 全双工
- 质量与安全门禁


## 1. 延迟预算

用户体感 ≈ 端点检测 + ASR 尾包 + LLM 首 token + TTS 首帧 + 网络。目标常：交互延迟 < 300–500ms。


In [ ]:
from dataclasses import dataclass


@dataclass
class LatencyBudget:
    vad_endpoint_ms: float = 120
    asr_tail_ms: float = 80
    llm_ttft_ms: float = 150
    tts_first_chunk_ms: float = 70
    net_ms: float = 40

    def total(self):
        return (self.vad_endpoint_ms + self.asr_tail_ms + self.llm_ttft_ms +
                self.tts_first_chunk_ms + self.net_ms)


b = LatencyBudget()
print('=== Latency Budget ===')
print(f'total={b.total():.0f}ms breakdown={b}')
# speculative: start LLM on partial ASR
b2 = LatencyBudget(asr_tail_ms=20, llm_ttft_ms=150)
print(f'with partial-ASR speculation total={b2.total():.0f}ms')
print(f'\nKey: Duplex UX is a latency budgeting problem more than a single-model accuracy problem.')


## 2. 半双工状态机：Listen → Think → Speak（可打断）

用户说话时暂停 TTS；检测到 barge-in 立即清空播放队列。


In [ ]:
class DuplexStateMachine:
    def __init__(self):
        self.state = 'listen'
        self.playback_q = []
        self.log = []

    def on_vad(self, speaking: bool):
        if speaking and self.state == 'speak':
            self.playback_q.clear()
            self.state = 'listen'
            self.log.append('barge_in_cancel_tts')
        elif speaking:
            self.state = 'listen'
        elif self.state == 'listen':
            self.state = 'think'

    def on_partial_transcript(self, text, conf):
        if self.state == 'think' and conf > 0.7 and len(text.split()) >= 3:
            self.log.append(('speculate', text))
            self.state = 'speak'
            self.playback_q.append(' proto-reply')

    def on_final(self, text):
        self.state = 'speak'
        self.playback_q.append(f'reply({text})')
        self.log.append(('final', text))


sm = DuplexStateMachine()
sm.on_vad(True)
sm.on_vad(False)
sm.on_partial_transcript('what is the weather in', 0.82)
sm.on_vad(True)  # user interrupts
sm.on_vad(False)
sm.on_final('what is the weather in berlin')
print('=== Duplex State Machine ===')
print('state=', sm.state, 'queue=', sm.playback_q)
print('log=', sm.log)
print(f'\nKey: Barge-in requires cancelable TTS and a single owner of playback state.')


## 3. 流式部分假设与投机回复

ASR 发出 partial hypothesis 时即可启动 LLM；若最终转写改写较大则丢弃草稿音频。类似解码投机。


In [ ]:
def should_commit_speculation(partial: str, final: str, thr=0.6):
    ps, fs = set(partial.lower().split()), set(final.lower().split())
    if not ps:
        return False
    overlap = len(ps & fs) / len(ps)
    return overlap >= thr


cases = [
    ('weather in ber', 'weather in berlin'),
    ('weather in ber', 'tell me a joke'),
]
print('=== Speculation Commit ===')
for p, f in cases:
    print(p, '->', f, 'commit=', should_commit_speculation(p, f))
print(f'\nKey: Speculative replies win latency only when partial ASR is stable enough.')


## 4. 全双工原生模型直觉

端到端模型同时消费音频帧并产出音频/文本 token，内部学习“何时听/何时说”。工程上仍需：
- AEC（回声消除）与设备播放参考信号  
- 打断标签 / 话轮样本  
- 安全：语音越狱、未成年人保护、录音同意


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)


class DuplexToy(nn.Module):
    '''Frame in -> (transcript logits, speak-gate, codec logits).'''
    def __init__(self, d_audio=16, vocab=30, codec=40):
        super().__init__()
        self.enc = nn.GRU(d_audio, 32, batch_first=True)
        self.asr = nn.Linear(32, vocab)
        self.gate = nn.Linear(32, 1)   # P(speak)
        self.tts = nn.Linear(32, codec)

    def forward(self, frames):
        h, _ = self.enc(frames)
        return self.asr(h), torch.sigmoid(self.gate(h)), self.tts(h)


model = DuplexToy()
frames = torch.randn(2, 50, 16)
asr_logits, gate, codec = model(frames)
# simulate turn-taking loss: speak when user silent
user_speaking = torch.zeros(2, 50, 1)
user_speaking[:, :20] = 1
speak_tgt = 1 - user_speaking
gate_loss = F.binary_cross_entropy(gate, speak_tgt)
print('=== Native Duplex Toy ===')
print(f'asr={tuple(asr_logits.shape)} gate={tuple(gate.shape)} codec={tuple(codec.shape)}')
print(f'turn-taking gate loss={gate_loss.item():.4f} mean_gate={gate.mean().item():.3f}')
print(f'\nKey: Full-duplex models must learn turn-taking, not only recognition and synthesis.')


## 课后思考题

1. 级联与全双工在可中断性、可运维性、多语种上如何取舍？
2. 如何评估 barge-in：误打断率与响应延迟如何平衡？
3. 部分 ASR 投机在噪声环境为何容易“说错话”？
4. 语音日志的合规同意与保留期限应如何设计？

---
> 本节涵盖了13.6 实时语音双工的核心概念与代码实现。建议结合实际项目需求，选择合适的技术方案，并通过实验验证不同方法的效果差异。
